# 03 · MCP 함수 직접 호출로 데이터 가져오기

**파이프라인 순서 3/4** — 에이전트/LLM 없이 **MCP 툴을 직접 호출**해 원시 데이터를 가져옵니다.
각 서버가 어떤 툴을 제공하고 무엇을 반환하는지 감을 잡는 단계입니다.

In [ ]:
# --- Bootstrap: locate llmwiki-pipeline/app and import the shared package ---
import sys
from pathlib import Path

def _find_app_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'app', base / 'llmwiki-pipeline' / 'app'):
            if (cand / 'pipeline' / '__init__.py').is_file():
                return cand
    raise RuntimeError('Could not locate llmwiki-pipeline/app — run this from inside the project.')

APP_DIR = _find_app_dir()
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
print('app dir on sys.path:', APP_DIR)

## 1. 토큰 + 툴 목록

In [ ]:
from pipeline import auth, config, mcp_client

tokens = {}
for key in config.SOURCE_KEYS:
    try:
        tokens[key] = auth.ensure_access_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')

tools_by_source = {}
for key, token in tokens.items():
    tools_by_source[key] = await mcp_client.list_tools(token, key)
    print(f'{key}: ' + ', '.join(t.name for t in tools_by_source[key]))

## 2. 검색/조회 툴 후보 찾기

이름에 search/list/get/find/message/mail 등이 포함된 읽기 툴을 추려봅니다.

In [ ]:
import re
READ_HINT = re.compile(r'(search|list|get|find|read|message|mail|chat|channel)', re.I)

for key, tools in tools_by_source.items():
    print(f'\n=== {key} 읽기 툴 후보 ===')
    for t in tools:
        if READ_HINT.search(t.name):
            print(' -', t.name)

## 3. 툴 직접 호출

아래에서 `SOURCE`, `TOOL_NAME`, `ARGS`를 위에서 확인한 실제 툴에 맞게 채워 실행하세요.
먼저 스키마를 출력해 필요한 인자를 확인합니다.

In [ ]:
import json

SOURCE = 'mail'          # 'mail' 또는 'teams'
TOOL_NAME = ''           # 위 목록에서 실제 툴 이름으로 채우세요 (예: 검색/리스트 툴)

if TOOL_NAME:
    tool = next((t for t in tools_by_source[SOURCE] if t.name == TOOL_NAME), None)
    assert tool, f'{TOOL_NAME} 을(를) {SOURCE} 툴 목록에서 찾을 수 없습니다.'
    print('입력 스키마:')
    print(json.dumps(mcp_client.tool_input_schema(tool), ensure_ascii=False, indent=2))
else:
    print('TOOL_NAME을 채운 뒤 다시 실행하세요. 위 후보 목록을 참고하세요.')

In [ ]:
# 스키마를 참고해 인자를 채우세요. 예) {'query': 'Postgres', 'top': 10}
ARGS = {}

if TOOL_NAME:
    result = await mcp_client.call_tool(tokens[SOURCE], SOURCE, TOOL_NAME, ARGS)
    text = mcp_client.content_to_text(result)
    print(text[:4000])

## 4. (선택) 여러 소스 동시 조회

`with_mcps`로 여러 소스에 동시에 연결해 각각 조회할 수도 있습니다.

In [ ]:
async def _peek(session):
    res = await session.list_tools()
    return [t.name for t in (res.tools or [])][:10]

clients, errors = None, None
results = await mcp_client.with_mcps(tokens, _peek)
print('per-source 결과:', results)

✅ 원하는 데이터를 직접 가져올 수 있음을 확인했습니다. 다음은 이 과정을 자연어로 자동화하는
**04_nl_aggregate_to_md** 입니다.